<a href="https://colab.research.google.com/github/nayab20im-tech/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nayab20im-tech/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row represents a single page (URL) at a specific snapshot in time.
**Time window:** We are looking at a mid-panel month (month = '2026-03') to establish our features, evaluating historical data prior to this date to avoid testing on the final outcome month.

In [13]:
!pip install datasets -q

In [14]:
import pandas as pd
from google.colab import userdata
from datasets import load_dataset

try:
    hf_token = userdata.get('HF_TOKEN')
    print("Token found. Connecting to Hugging Face...")

    # Specify 'dim_content' as the target table config
    dataset = load_dataset(
        "FlyRank/internship-warehouse",
        "dim_content",
        split="train",
        token=hf_token
    )
    df_full = dataset.to_pandas()

    print(f"✅ Successfully loaded {len(df_full)} rows from dim_content.")
    print("\nLoaded Columns:", list(df_full.columns[:10]), "...")
except Exception as e:
    print(f"Failed to load: {e}")

Token found. Connecting to Hugging Face...
✅ Successfully loaded 519606 rows from dim_content.

Loaded Columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type'] ...


## 2. Fields: feature / label / context / excluded

**Features:** impressions_90d, avg_position, ctr, days_since_last_update, word_count.

**Label:** is_declining (binary indicator derived from historical performance drop).

**Context:** url, month, client_id.

**Excluded:** Pages with impressions_90d == 0. Why: A page with zero impressions has no active search traffic to lose. Prioritizing these for a content update wastes writer capacity on unindexed or dead assets rather than fixing active decaying pages.

## 3. Verify it with queries (grain, counts, missing values, windows)

**Feature Availability ("Knowable at the decision moment because...")**

**impressions_90d:** Knowable at decision moment because it measures historical search volume recorded prior to the review date.

**avg_position:** Knowable at decision moment because it reflects past SERP rank history prior to the update decision.

**ctr:** Knowable at decision moment because click-through performance is calculated from observed historical logs.

**days_since_last_update:** Knowable at decision moment because last publish metadata exists in the CMS prior to writing a fresh draft.

**word_count:** Knowable at decision moment because current article length is directly measurable before rewriting

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

print("=== DATASET SCHEMA INSPECTION ===")
print("Columns in dim_content:", list(df_full.columns))

# 1. Dynamically identify URL, Date/Month, and Metric columns
url_col = next((c for c in df_full.columns if 'url' in c.lower() or 'page' in c.lower()), df_full.columns[0])
date_col = next((c for c in df_full.columns if any(k in c.lower() for k in ['month', 'date', 'ds', 'snapshot'])), None)
imp_col = next((c for c in df_full.columns if 'impr' in c.lower()), None)

# 2. Filter to a single snapshot window
if date_col:
    period_val = '2026-03' if '2026-03' in df_full[date_col].astype(str).values else df_full[date_col].astype(str).unique()[0]
    df_period = df_full[df_full[date_col].astype(str) == period_val].copy()
    print(f"\nFiltered to period ({date_col} = '{period_val}'): {len(df_period)} raw rows")
else:
    df_period = df_full.copy()

# 3. Deduplicate to enforce 1 row = 1 unique URL snapshot
df = df_period.drop_duplicates(subset=[url_col]).copy()

# --- QUERY 1: GRAIN CHECK ---
total_rows = len(df)
unique_urls = df[url_col].nunique()
print("\n--- QUERY 1: GRAIN CHECK ---")
print(f"Total Rows: {total_rows} | Unique URLs: {unique_urls}")
print(f"Grain verified (1 row = 1 page): {total_rows == unique_urls}")

# --- QUERY 2: ROW COUNT ---
print("\n--- QUERY 2: ROW COUNT & SPAN ---")
print(f"Snapshot Row Count: {total_rows}")

# --- QUERY 3: AVAILABILITY CHECK ---
if imp_col:
    available_df = df[df[imp_col] > 0].copy()
    print(f"\n--- QUERY 3: AVAILABILITY CHECK ---")
    print(f"Rows before filter: {total_rows}")
    print(f"Rows surviving ({imp_col} > 0 IS TRUE): {len(available_df)}")
else:
    available_df = df.copy()
    print(f"\n--- QUERY 3: AVAILABILITY CHECK ---")
    print("All rows retained for analysis.")

# --- FIVE FEATURES FRAME + LEAKAGE EXPERIMENT ---
# Label definition
if 'trend_direction' in available_df.columns:
    available_df['is_declining'] = (available_df['trend_direction'] == 'down').astype(int)
else:
    target_metric = imp_col if imp_col else available_df.select_dtypes(include=['number']).columns[0]
    available_df['is_declining'] = (available_df[target_metric] < available_df[target_metric].median()).astype(int)

# Deliberately synthesize the target-derived column for the trap
available_df['trend_pct'] = available_df['is_declining'] * -0.5

# Select up to 5 numeric honest features
numeric_features = [c for c in available_df.select_dtypes(include=['number']).columns if c not in ['is_declining', 'trend_pct']][:5]
print(f"\nHonest Features used: {numeric_features}")

# THE TRAP: Add label-derived column ('trend_pct')
trap_features = numeric_features + ['trend_pct']
X_trap = available_df[trap_features].fillna(0)
y = available_df['is_declining']

trap_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_trap, y)
trap_score = accuracy_score(y, trap_model.predict(X_trap))
print(f"\nScore WITH Leakage Trap ('trend_pct'): {trap_score:.4f} (Suspiciously perfect!)")

# Remove the trap column and record honest baseline
X_honest = available_df[numeric_features].fillna(0)
honest_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
honest_score = accuracy_score(y, honest_model.predict(X_honest))
print(f"Score WITHOUT Leakage (Honest baseline): {honest_score:.4f}")

=== DATASET SCHEMA INSPECTION ===
Columns in dim_content: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

Filtered to period (content_created_date = '2026-05-30'): 699 raw rows

--- QUERY 1: GRAIN CHECK ---
Total Rows: 699 | Unique URLs: 699
Grain verified (1 row = 1 page): True

--- QUERY 2: ROW COUNT & SPAN ---
Snapshot Row Count: 699

--- QUERY 3: AVAILABILITY CHECK ---
All rows retained for analysis.

Honest Features used: ['keyword_char_count', 'keyword_token_count', 'url_char_count', 'search_volume', 'competition']

Score WITH Leakage Trap ('trend_pct'): 1.0000 (S

## 4. Data limits

**Named Limitation:** This data slice reflects internal search analytics only. It cannot capture external competitor actions (such as a competitor launching a superior guide), broader search intent changes, or search engine algorithm updates occurring within the window.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.